In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
%pip install pypdf

In [0]:
policy_bronze_df=spark.read.table("catalog1.bronze_layer.bronze_policy_table")
claims_bronze_df=spark.read.table("catalog1.bronze_layer.bronze_claims_table")


In [0]:
from pypdf import PdfReader
import io

policy_text_data = []

rows = policy_bronze_df.collect()

for row in rows:
    
    file_name = row.path.split("/")[-1]
    
    # Convert binary to file-like object
    pdf_stream = io.BytesIO(row.content)
    
    reader = PdfReader(pdf_stream)
    text = ""
    
    for page in reader.pages:
        extracted = page.extract_text()
        if extracted:
            text += extracted
    
    policy_text_data.append((file_name, text))

policy_df = spark.createDataFrame(policy_text_data, ["file_name", "text"])


In [0]:
from pyspark.sql.functions import regexp_replace, col

silver_policy_df = policy_df.withColumn(
    "clean_text",
    regexp_replace(col("text"), "\\n", " ")
)



In [0]:
silver_policy_df.display()

In [0]:
from pypdf import PdfReader
import io
claim_text_data = []

rows = claims_bronze_df.collect()

for row in rows:
    
    file_name = row.path.split("/")[-1]
    
    pdf_stream = io.BytesIO(row.content)
    
    reader = PdfReader(pdf_stream)
    text = ""
    for page in reader.pages:
        extracted = page.extract_text()
        if extracted:
            text += extracted
    
    claim_text_data.append((file_name, text))

claim_df = spark.createDataFrame(claim_text_data, ["file_name", "text"])


In [0]:
from pyspark.sql.functions import regexp_replace, col

silver_claim_df = claim_df.withColumn(
    "clean_text",
    regexp_replace(col("text"), "\\n", " ")
)

In [0]:
silver_claim_df.display()

In [0]:
%sql
create database if not exists catalog1.silver_layer;

In [0]:
silver_claim_df.select("file_name", "clean_text") \
    .write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("catalog1.silver_layer.silver_claim_table")

In [0]:
silver_policy_df.select("file_name", "clean_text") \
    .write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("catalog1.silver_layer.silver_policy_table")

In [0]:
%sql
select * from catalog1.silver_layer.silver_claim_table limit 5


In [0]:
%sql
select * from catalog1.silver_layer.silver_policy_table limit 5